# SQLForge — 03: GRPO (Colab, T4)

RL on top of the SFT adapter with execution-based rewards. Requires the SFT
adapter from notebook 02 in Drive (`outputs/sft/adapter`).
Expect 2–3 T4 sessions; training resumes from the last checkpoint automatically.

Background reading: `docs/grpo-explained.md`.

In [ ]:
REPO_URL = "https://github.com/YOUR_USERNAME/sqlforge.git"  # <-- set me
from google.colab import drive
drive.mount('/content/drive')
!git clone $REPO_URL
%cd sqlforge
!pip install -q uv && uv sync --group ml --group train
!ln -sfn /content/drive/MyDrive/sqlforge/outputs outputs && ls outputs/sft/adapter

In [ ]:
!uv run python scripts/download_spider.py
!uv run python scripts/build_dataset.py

In [ ]:
!uv run python training/grpo.py

In [ ]:
# Reward curves: each component separately, plus KL and completion length.
import json

import matplotlib.pyplot as plt

logs = json.load(open('outputs/grpo/log_history.json'))
fig, axes = plt.subplots(1, 3, figsize=(16, 4))
for key in [k for k in logs[0] if k.startswith('rewards/')]:
    pts = [(e['step'], e[key]) for e in logs if key in e]
    axes[0].plot([p[0] for p in pts], [p[1] for p in pts], label=key.split('/')[1])
axes[0].set_title('reward components'); axes[0].legend(fontsize=8)
for ax, key, title in [(axes[1], 'kl', 'KL vs reference'),
                        (axes[2], 'completions/mean_length', 'completion length')]:
    pts = [(e['step'], e[key]) for e in logs if key in e]
    ax.plot([p[0] for p in pts], [p[1] for p in pts]); ax.set_title(title)
plt.savefig('docs/assets/grpo_rewards.png', dpi=150, bbox_inches='tight'); plt.show()

In [ ]:
# Evaluate GRPO adapter, then inspect 30 outputs for reward hacking.
!uv run python scripts/run_eval.py --name grpo-full --adapter outputs/grpo/adapter --batch-size 16 --log-results
!uv run python scripts/inspect_outputs.py --eval-json outputs/eval/grpo-full.json